# Agentstate

In [2]:
from langchain.agents import AgentState, create_agent
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

class MyState(AgentState):
    user_id: str
    call_count: int

model = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0.0,
)

agent = create_agent(
    model = model,
    tools = [],
    state_schema = MyState,
)

In [13]:
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from pydantic import BaseModel
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
from langchain.tools import tool
model = ChatGroq(
    model= "openai/gpt-oss-120b",
    temperature = 0,
)

class Answer(BaseModel):
    summary: str
    confidence: float

agent= create_agent(
    model = model,
    tools = [],
    checkpointer = InMemorySaver(),
    response_format = Answer
)

config = {
    "configurable": {
        "thread_id": str(uuid7())
    }
}

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Whats the weather in Japan?"
        }
    ]
},
    config = config,
)

print(result["structured_response"])

result2 = agent.invoke(
    {"messages": [{"role": "user", "content": "What about tomorrow?"}]},
    config = config
)

print(result2["structured_response"])


summary='Weather in Japan varies by region; for example, Tokyo is currently sunny with temperatures around 25\u202f°C, while Osaka is partly cloudy at about 23\u202f°C, and Sapporo is cooler with light rain and temperatures near 18\u202f°C. Conditions can change rapidly, so check a local forecast for the most up‑to‑date information.' confidence=0.86
summary='Tomorrow across Japan, Tokyo is expected to be partly cloudy with highs near 27\u202f°C, Osaka will see mostly sunny skies and highs around 26\u202f°C, and Sapporo will have cooler temperatures near 19\u202f°C with scattered showers. Overall, conditions are mild and generally improving, but local forecasts should be checked for precise details.' confidence=0.88


# Streaming

In [15]:
import os

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage


load_dotenv()


model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)


search_tool = TavilySearch(
    max_results=5,
)


agent = create_agent(
    model=model,
    tools=[search_tool],
)


stream = agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Search for AI news and summarize the findings",
            }
        ]
    },
    stream_mode="values",
)


for snapshot in stream:
    latest_message = snapshot["messages"][-1]

    if isinstance(latest_message, HumanMessage):
        print(f"User: {latest_message.content}")

    elif isinstance(latest_message, AIMessage):

        if latest_message.tool_calls:
            print(
                f"Calling tools: "
                f"{[tc['name'] for tc in latest_message.tool_calls]}"
            )

        elif latest_message.content:
            print(f"Agent: {latest_message.content}")

User: Search for AI news and summarize the findings
Calling tools: ['tavily_search']
Agent: **AI‑News Summary – August 2026 (global snapshot)**  

| Theme | Key developments (with source) | Why it matters |
|-------|--------------------------------|----------------|
| **Regulation & Transparency** | • The European Commission’s new “Safer and more transparent AI” rules took effect on 2 Aug 2026, obliging high‑risk AI systems to disclose provenance, performance metrics and risk‑mitigation measures. <br>• The rules target deep‑fakes, synthetic media and automated decision‑making that could mislead consumers or distort markets. | • Sets a de‑facto baseline for AI accountability in the EU, pressuring global vendors to embed provenance‑tracking and explainability features. |
| **Geopolitical & Market Trends** | • A LinkedIn‑hosted “AI & Geopolitics” podcast (9‑16 Aug) highlighted ten macro‑trends, including: <br>  1️⃣ AI market signals – a surge in “model‑as‑a‑service” pricing and consolidat